# Reconstruction Baselines

With the pipeline verified in Notebook 01, we now train a U-Net on real fastMRI knee data. The goal is not state-of-the-art reconstruction, but a model that produces reconstructions good enough to contain meaningful hallucinations for detection in later notebooks.

**What this notebook covers:**
1. Training a U-Net on 150 volumes (~5400 slices) with mixed precision and cosine annealing
2. Comparison of IFFT vs U-Net at 4x and 8x acceleration
3. Quantitative evaluation across the full validation set (199 volumes)

**Critical design choice:** We train against `reconstruction_esc` (emulated singlecoil), not `reconstruction_rss` (root-sum-of-squares from multicoil). RSS was computed from multicoil data that our singlecoil k-space cannot represent, creating a hard ceiling of ~26 dB. ESC matches our IFFT at 155 dB, confirming pipeline correctness (Notebook 01). Training against ESC means the U-Net only learns to fill missing k-space, which is the actual task.

**Checkpoint:** The trained model is saved to Google Drive and reused in Notebooks 03 through 05.

## Setup

Requires a **GPU runtime** (L4 recommended). Training takes ~90 minutes with mixed precision on L4.

**Data paths:**
- Validation: `/content/drive/MyDrive/fastmri/singlecoil_val/` (synced to local SSD)
- Training: `/content/drive/MyDrive/fastmri/singlecoil_train/` (150 volumes synced locally)

In [ ]:
!pip install fastmri h5py scikit-image pyyaml tqdm -q

import os, time, gc, json
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import h5py
import yaml
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

FASTMRI_DATA_DIR = os.environ.get("FASTMRI_DATA_DIR", "/content/data/")
os.makedirs(FASTMRI_DATA_DIR, exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("configs", exist_ok=True)

DEFAULT_CONFIG = {
    'data': {
        'dataset': 'fastmri_knee_singlecoil',
        'data_dir': FASTMRI_DATA_DIR,
        'acceleration': 4,
        'center_fraction': 0.08,
        'mask_types': ['random', 'equispaced', 'gaussian', 'poisson_disc'],
        'image_size': [320, 320],
    },
    'model': {
        'architecture': 'unet',
        'channels': [32, 64, 128, 256],
        'dropout_p': 0.05,
    },
    'training': {
        'epochs': 50, 'lr': 1e-3, 'batch_size': 8, 'loss': 'l1', 'seed': 42,
    },
    'figures': {
        'dpi': 300, 'font_size_min': 12,
        'cmap_magnitude': 'gray', 'cmap_kspace': 'viridis',
        'cmap_error': 'RdBu_r', 'cmap_detection': 'hot',
        'background': 'white',
    },
}

with open('configs/default.yaml', 'w') as f:
    yaml.dump(DEFAULT_CONFIG, f, default_flow_style=False, sort_keys=False)
with open('configs/default.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

mpl.rcParams.update({
    'figure.dpi': cfg['figures']['dpi'],
    'savefig.dpi': cfg['figures']['dpi'],
    'font.size': cfg['figures']['font_size_min'],
    'axes.titlesize': 13, 'axes.labelsize': 12,
    'figure.facecolor': cfg['figures']['background'],
    'savefig.facecolor': cfg['figures']['background'],
    'savefig.bbox': 'tight',
})

torch.manual_seed(cfg['training']['seed'])
np.random.seed(cfg['training']['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed(cfg['training']['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# --------------- Mount Drive & sync data ---------------
from google.colab import drive
import subprocess

drive.mount('/content/drive', force_remount=True)

# Sync validation set to local SSD
!mkdir -p /content/data/singlecoil_val
subprocess.run(['rsync', '-a', '--ignore-existing',
                '/content/drive/MyDrive/fastmri/singlecoil_val/',
                '/content/data/singlecoil_val/'],
               capture_output=True)
n_val = len(list(Path('/content/data/singlecoil_val').glob('*.h5')))
print(f"Val volumes (local): {n_val}")

# Sync 150 train volumes to local SSD (~12 GB)
!mkdir -p /content/data/singlecoil_train
train_drive = '/content/drive/MyDrive/fastmri/singlecoil_train/'
train_local = '/content/data/singlecoil_train/'

h5_train_files = sorted(Path(train_drive).glob('*.h5'))[:150]
for i, f in enumerate(h5_train_files):
    dst = Path(train_local) / f.name
    if not dst.exists():
        subprocess.run(['cp', str(f), train_local], capture_output=True)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/150 checked")

n_train = len(list(Path(train_local).glob('*.h5')))
print(f"Train volumes (local): {n_train}")

## 1. Utilities

All Fourier functions, the MRI forward operator, mask generation, metrics, and the U-Net architecture are copied from Notebook 01 to keep this notebook self-contained. The U-Net has `Dropout2d(p=0.05)` baked in, which is inactive during normal inference but will be activated for MC Dropout in Notebook 05.

In [ ]:
# ---- Fourier utilities (norm='ortho' everywhere) ----

def to_kspace(image: torch.Tensor) -> torch.Tensor:
    x = image.to(torch.complex64) if not image.is_complex() else image
    return torch.fft.fftshift(
        torch.fft.fft2(torch.fft.ifftshift(x, dim=(-2, -1)), dim=(-2, -1), norm='ortho'),
        dim=(-2, -1))

def from_kspace(kspace: torch.Tensor) -> torch.Tensor:
    return torch.fft.fftshift(
        torch.fft.ifft2(torch.fft.ifftshift(kspace, dim=(-2, -1)), dim=(-2, -1), norm='ortho'),
        dim=(-2, -1))

def center_crop(image: torch.Tensor, target_shape: tuple) -> torch.Tensor:
    h, w = image.shape[-2], image.shape[-1]
    th, tw = target_shape
    return image[..., (h-th)//2:(h-th)//2+th, (w-tw)//2:(w-tw)//2+tw]

def reconstruct_and_crop(kspace, target_shape=(320, 320)):
    return center_crop(torch.abs(from_kspace(kspace)), target_shape)


# ---- MRI operator ----

class CartesianMRIOperator:
    def __init__(self, mask):
        self.mask = mask.float()
    def forward(self, image):
        return to_kspace(image) * self.mask
    def adjoint(self, kspace):
        return from_kspace(kspace * self.mask)
    def normal(self, image):
        return self.adjoint(self.forward(image))
    def null_space_project(self, image):
        return image.to(torch.complex64) - self.normal(image)


# ---- Masks ----

def create_mask(shape, acceleration=4, center_fraction=0.08, mask_type='random', seed=None):
    rng = np.random.RandomState(seed) if seed is not None else np.random.RandomState()
    H, W = shape[-2], shape[-1]
    num_center = int(W * center_fraction)
    num_total = max(int(W / acceleration), num_center)
    num_outer = num_total - num_center

    mask = np.zeros(W, dtype=np.float32)
    center_start = (W - num_center) // 2
    mask[center_start:center_start + num_center] = 1.0
    outer_indices = np.where(mask == 0)[0]

    if mask_type == 'random':
        chosen = rng.choice(outer_indices, size=min(num_outer, len(outer_indices)), replace=False)
        mask[chosen] = 1.0
    elif mask_type == 'equispaced':
        if num_outer > 0 and len(outer_indices) > 0:
            step = max(len(outer_indices) // num_outer, 1)
            offset = rng.randint(0, step) if step > 1 else 0
            mask[outer_indices[offset::step][:num_outer]] = 1.0
    elif mask_type == 'gaussian':
        sigma = W / 6.0
        probs = np.exp(-0.5 * ((outer_indices - W/2.0) / sigma) ** 2)
        probs /= probs.sum()
        chosen = rng.choice(outer_indices, size=min(num_outer, len(outer_indices)), replace=False, p=probs)
        mask[chosen] = 1.0
    elif mask_type == 'poisson_disc':
        if num_outer > 0 and len(outer_indices) > 0:
            min_dist = max(len(outer_indices) / (num_outer * 1.5), 1.0)
            selected = []
            candidates = list(outer_indices); rng.shuffle(candidates)
            for c in candidates:
                if len(selected) >= num_outer: break
                if all(abs(c - s) >= min_dist for s in selected):
                    selected.append(c)
            if len(selected) < num_outer:
                remaining = [c for c in outer_indices if c not in selected]
                rng.shuffle(remaining)
                selected += remaining[:num_outer - len(selected)]
            mask[np.array(selected)] = 1.0
    return torch.from_numpy(mask).unsqueeze(0)


# ---- Metrics ----

def compute_metrics(gt, recon):
    gt = np.abs(gt).astype(np.float64)
    recon = np.abs(recon).astype(np.float64)
    data_range = gt.max() - gt.min()
    if data_range == 0:
        return {'psnr': float('inf'), 'ssim': 1.0, 'nmse': 0.0}
    return {
        'psnr': peak_signal_noise_ratio(gt, recon, data_range=data_range),
        'ssim': structural_similarity(gt, recon, data_range=data_range),
        'nmse': np.sum((gt - recon)**2) / np.sum(gt**2),
    }


# ---- U-Net ----

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, channels=(32, 64, 128, 256), dropout_p=0.05):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools = nn.ModuleList()
        self.decoders = nn.ModuleList()
        self.upconvs = nn.ModuleList()
        self.dropout = nn.Dropout2d(p=dropout_p)

        in_ch = 1
        for ch in channels:
            self.encoders.append(ConvBlock(in_ch, ch))
            self.pools.append(nn.MaxPool2d(2))
            in_ch = ch
        self.bottleneck = ConvBlock(channels[-1], channels[-1] * 2)
        for ch in reversed(channels):
            self.upconvs.append(nn.ConvTranspose2d(ch * 2, ch, 2, stride=2))
            self.decoders.append(ConvBlock(ch * 2, ch))
        self.final = nn.Conv2d(channels[0], 1, 1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x)
            x = pool(x); x = self.dropout(x)
        x = self.bottleneck(x)
        for upconv, dec, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = upconv(x)
            if x.shape != skip.shape:
                x = nn.functional.pad(x, [0, skip.shape[3]-x.shape[3], 0, skip.shape[2]-x.shape[2]])
            x = torch.cat([x, skip], dim=1)
            x = dec(x); x = self.dropout(x)
        return self.final(x)

print(f"U-Net: {sum(p.numel() for p in UNet().parameters()):,} parameters")

## 2. Dataset

The dataset handles variable k-space widths across volumes (368, 372, etc.) by performing IFFT, center crop, and normalization inside `__getitem__`, so every sample comes out as a fixed `(1, 320, 320)` tensor in [0, 1].

Training uses random masks (different every epoch, acting as data augmentation) and random horizontal flips. Validation uses fixed masks for reproducibility.

In [ ]:
class FastMRIDataset(Dataset):
    """fastMRI singlecoil with per-sample normalization and optional augmentation."""

    def __init__(self, data_dir, acceleration=4, center_fraction=0.08,
                 mask_type='random', target_type='reconstruction_esc',
                 fixed_masks=False, max_volumes=None, augment=False):
        self.data_dir = Path(data_dir)
        self.acceleration = acceleration
        self.center_fraction = center_fraction
        self.mask_type = mask_type
        self.target_type = target_type
        self.fixed_masks = fixed_masks
        self.augment = augment

        self.examples = []
        h5_files = sorted(self.data_dir.glob("*.h5"))
        if max_volumes:
            h5_files = h5_files[:max_volumes]

        skipped = 0
        for h5_path in h5_files:
            try:
                with h5py.File(h5_path, "r") as f:
                    if "kspace" not in f or self.target_type not in f:
                        skipped += 1; continue
                    self.examples += [(h5_path, i) for i in range(f["kspace"].shape[0])]
            except Exception:
                skipped += 1

        n_vols = len(set(p for p, _ in self.examples))
        print(f"FastMRIDataset: {len(self.examples)} slices from {n_vols} volumes "
              f"({self.data_dir.name})")
        if skipped:
            print(f"  Skipped {skipped} volumes")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        h5_path, slice_idx = self.examples[idx]
        with h5py.File(h5_path, "r") as f:
            kspace = torch.from_numpy(f["kspace"][slice_idx].copy())
            target = torch.from_numpy(f[self.target_type][slice_idx].copy()).float()

        seed = idx if self.fixed_masks else None
        mask = create_mask(kspace.shape, acceleration=self.acceleration,
                           center_fraction=self.center_fraction,
                           mask_type=self.mask_type, seed=seed)

        # IFFT -> magnitude -> crop -> normalize
        mag = center_crop(torch.abs(from_kspace(kspace * mask)), (320, 320))
        mag_min, mag_max = mag.min(), mag.max()
        input_img = ((mag - mag_min) / (mag_max - mag_min + 1e-8)).unsqueeze(0)

        t_min, t_max = target.min(), target.max()
        target_norm = ((target - t_min) / (t_max - t_min + 1e-8)).unsqueeze(0)

        if self.augment and torch.rand(1).item() > 0.5:
            input_img = torch.flip(input_img, dims=[-1])
            target_norm = torch.flip(target_norm, dims=[-1])

        return input_img, target_norm


train_dataset = FastMRIDataset(
    '/content/data/singlecoil_train/', acceleration=4,
    target_type='reconstruction_esc', fixed_masks=False,
    max_volumes=150, augment=True)

val_dataset = FastMRIDataset(
    '/content/data/singlecoil_val/', acceleration=4,
    target_type='reconstruction_esc', fixed_masks=True, augment=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

print(f"\nTrain: {len(train_dataset)} slices, {len(train_loader)} batches/epoch")
print(f"Val:   {len(val_dataset)} slices, {len(val_loader)} batches")

## 3. Training

L1 loss with Adam optimizer and cosine annealing learning rate schedule (1e-3 down to 1e-5 over 50 epochs). Mixed precision (AMP) roughly doubles throughput on L4.

The best checkpoint is saved to Drive based on validation loss. Training 150 volumes is deliberately not the full 973: a weaker U-Net produces more hallucinations, giving a stronger signal for the detection methods in Notebooks 03 through 05.

In [ ]:
model = UNet(channels=tuple(cfg['model']['channels']),
             dropout_p=cfg['model']['dropout_p']).to(device)

NUM_EPOCHS = 50
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)
criterion = nn.L1Loss()
scaler = torch.amp.GradScaler('cuda')

os.makedirs('/content/drive/MyDrive/fastmri/checkpoints', exist_ok=True)

train_losses, val_losses, lr_history = [], [], []
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    t0 = time.time()

    # Train
    model.train()
    running = 0.0
    for input_img, target_img in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        input_img, target_img = input_img.to(device), target_img.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model(input_img), target_img)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item()
    train_loss = running / len(train_loader)
    train_losses.append(train_loss)

    # Validate
    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for input_img, target_img in val_loader:
            input_img, target_img = input_img.to(device), target_img.to(device)
            with torch.amp.autocast('cuda'):
                val_running += criterion(model(input_img), target_img).item()
    val_loss = val_running / len(val_loader)
    val_losses.append(val_loss)

    lr_history.append(optimizer.param_groups[0]['lr'])
    scheduler.step()

    improved = " *BEST*" if val_loss < best_val_loss else ""
    print(f"  Train L1: {train_loss:.4f} | Val L1: {val_loss:.4f} | "
          f"LR: {lr_history[-1]:.1e} | {time.time()-t0:.0f}s{improved}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss, 'val_loss': val_loss,
            'train_losses': train_losses, 'val_losses': val_losses,
        }, '/content/drive/MyDrive/fastmri/checkpoints/unet_4x_v2_best.pt')

    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'train_losses': train_losses, 'val_losses': val_losses,
        }, f'/content/drive/MyDrive/fastmri/checkpoints/unet_4x_v2_epoch{epoch+1}.pt')

print(f"\nBest val L1: {best_val_loss:.4f} at epoch {np.argmin(val_losses)+1}")

# Load best for evaluation
ckpt = torch.load('/content/drive/MyDrive/fastmri/checkpoints/unet_4x_v2_best.pt',
                   map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best model (epoch {ckpt['epoch']+1})")

## 4. Training Curves

Three views of the training dynamics: the loss convergence, validation loss detail with best epoch marked, and the cosine annealing learning rate schedule.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
epochs_range = range(1, len(train_losses) + 1)

axes[0].plot(epochs_range, train_losses, 'b-', lw=1.5, label='Train')
axes[0].plot(epochs_range, val_losses, 'r-', lw=1.5, label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('L1 Loss')
axes[0].set_title('(a) Training and Validation Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, val_losses, 'r-o', markersize=2, lw=1.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val L1 Loss')
axes[1].set_title('(b) Validation Loss (detail)')
best_ep = np.argmin(val_losses) + 1
axes[1].axvline(x=best_ep, color='green', ls='--', alpha=0.7, label=f'Best: epoch {best_ep}')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, lr_history, 'g-', lw=1.5)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate')
axes[2].set_title('(c) Cosine Annealing Schedule')
axes[2].set_yscale('log'); axes[2].grid(True, alpha=0.3)

n_vols = len(set(p for p, _ in train_dataset.examples))
fig.suptitle(f'U-Net Training: {n_vols} volumes, {len(train_losses)} epochs, AMP, ESC target', fontsize=14)
plt.tight_layout()
plt.savefig('figures/02_training_curves.png', bbox_inches='tight')
plt.show()

![Figure 1](knee_mri_final_2_figs/fig_001.png)


## 5. Qualitative Results at 4x Acceleration

Three slices from different validation volumes. Each row shows the zero-filled IFFT (physics baseline), the U-Net reconstruction, the ESC ground truth, and the U-Net error map. The error maps use a diverging colormap (red = overestimate, blue = underestimate) to show spatial structure in the reconstruction errors. These structured errors are the hallucinations we will characterize in Notebook 03.

In [ ]:
val_h5s = sorted(Path('/content/data/singlecoil_val/').glob('*.h5'))
sample_indices = [0, len(val_h5s)//3, 2*len(val_h5s)//3]

fig, axes = plt.subplots(3, 4, figsize=(18, 13))

for row, vol_idx in enumerate(sample_indices):
    with h5py.File(val_h5s[vol_idx], 'r') as f:
        mid = f['kspace'].shape[0] // 2
        kspace_full = torch.from_numpy(f['kspace'][mid].copy())
        target_esc = torch.from_numpy(f['reconstruction_esc'][mid].copy()).float()

    mask = create_mask(kspace_full.shape, acceleration=4, mask_type='random', seed=42+row)
    ifft_recon = center_crop(torch.abs(from_kspace(kspace_full * mask)), (320, 320))
    ifft_norm = (ifft_recon - ifft_recon.min()) / (ifft_recon.max() - ifft_recon.min() + 1e-8)
    target_norm = (target_esc - target_esc.min()) / (target_esc.max() - target_esc.min() + 1e-8)

    with torch.no_grad():
        unet_out = model(ifft_norm.unsqueeze(0).unsqueeze(0).to(device)).cpu().squeeze()

    m_ifft = compute_metrics(target_norm.numpy(), ifft_norm.numpy())
    m_unet = compute_metrics(target_norm.numpy(), unet_out.numpy())

    axes[row, 0].imshow(ifft_norm.numpy(), cmap='gray')
    axes[row, 0].set_title(f'PSNR={m_ifft["psnr"]:.1f} dB' if row > 0
                           else f'(a) IFFT 4x\nPSNR={m_ifft["psnr"]:.1f} dB')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(unet_out.numpy(), cmap='gray')
    axes[row, 1].set_title(f'PSNR={m_unet["psnr"]:.1f} dB' if row > 0
                           else f'(b) U-Net 4x\nPSNR={m_unet["psnr"]:.1f} dB')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(target_norm.numpy(), cmap='gray')
    axes[row, 2].set_title('(c) Ground Truth (ESC)' if row == 0 else '')
    axes[row, 2].axis('off')

    error = unet_out.numpy() - target_norm.numpy()
    vmax = np.percentile(np.abs(error), 99)
    im = axes[row, 3].imshow(error, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[row, 3].set_title('(d) U-Net Error' if row == 0 else '')
    axes[row, 3].axis('off')
    plt.colorbar(im, ax=axes[row, 3], fraction=0.046, pad=0.04)

    print(f"Vol {vol_idx}: IFFT {m_ifft['psnr']:.1f} dB -> U-Net {m_unet['psnr']:.1f} dB "
          f"(+{m_unet['psnr']-m_ifft['psnr']:.1f})")

fig.suptitle('4x Acceleration: IFFT vs U-Net vs Ground Truth', fontsize=15)
plt.tight_layout()
plt.savefig('figures/02_qualitative_4x.png', bbox_inches='tight')
plt.show()

![Figure 2](knee_mri_final_2_figs/fig_002.png)


## 6. Qualitative Results at 8x Acceleration

Same U-Net (trained at 4x only) applied to 8x undersampled data. With half the k-space lines compared to 4x, the null space is larger. The network must fabricate more content to produce a plausible image. This is exactly the regime Notebook 03 will analyze: more missing data means more hallucination.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 13))

for row, vol_idx in enumerate(sample_indices):
    with h5py.File(val_h5s[vol_idx], 'r') as f:
        mid = f['kspace'].shape[0] // 2
        kspace_full = torch.from_numpy(f['kspace'][mid].copy())
        target_esc = torch.from_numpy(f['reconstruction_esc'][mid].copy()).float()

    mask = create_mask(kspace_full.shape, acceleration=8, mask_type='random', seed=42+row)
    ifft_recon = center_crop(torch.abs(from_kspace(kspace_full * mask)), (320, 320))
    ifft_norm = (ifft_recon - ifft_recon.min()) / (ifft_recon.max() - ifft_recon.min() + 1e-8)
    target_norm = (target_esc - target_esc.min()) / (target_esc.max() - target_esc.min() + 1e-8)

    with torch.no_grad():
        unet_out = model(ifft_norm.unsqueeze(0).unsqueeze(0).to(device)).cpu().squeeze()

    m_ifft = compute_metrics(target_norm.numpy(), ifft_norm.numpy())
    m_unet = compute_metrics(target_norm.numpy(), unet_out.numpy())

    axes[row, 0].imshow(ifft_norm.numpy(), cmap='gray')
    axes[row, 0].set_title(f'PSNR={m_ifft["psnr"]:.1f} dB' if row > 0
                           else f'(a) IFFT 8x\nPSNR={m_ifft["psnr"]:.1f} dB')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(unet_out.numpy(), cmap='gray')
    axes[row, 1].set_title(f'PSNR={m_unet["psnr"]:.1f} dB' if row > 0
                           else f'(b) U-Net 8x\nPSNR={m_unet["psnr"]:.1f} dB')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(target_norm.numpy(), cmap='gray')
    axes[row, 2].set_title('(c) Ground Truth (ESC)' if row == 0 else '')
    axes[row, 2].axis('off')

    error = unet_out.numpy() - target_norm.numpy()
    vmax = np.percentile(np.abs(error), 99)
    im = axes[row, 3].imshow(error, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[row, 3].set_title('(d) U-Net Error' if row == 0 else '')
    axes[row, 3].axis('off')
    plt.colorbar(im, ax=axes[row, 3], fraction=0.046, pad=0.04)

    print(f"Vol {vol_idx}: IFFT {m_ifft['psnr']:.1f} dB -> U-Net {m_unet['psnr']:.1f} dB "
          f"(+{m_unet['psnr']-m_ifft['psnr']:.1f})")

fig.suptitle('8x Acceleration: IFFT vs U-Net (trained at 4x) vs Ground Truth', fontsize=15)
plt.tight_layout()
plt.savefig('figures/02_qualitative_8x.png', bbox_inches='tight')
plt.show()

![Figure 3](knee_mri_final_2_figs/fig_003.png)


## 7. Quantitative Evaluation

Full evaluation on the validation set (199 volumes, middle 50% of slices to skip uninformative edge slices). Both IFFT and U-Net are evaluated at 4x and 8x acceleration, all against `reconstruction_esc`.

In [ ]:
def evaluate_method(val_dir, model, device, acceleration, max_volumes=None):
    """Evaluate IFFT and U-Net across val set. Returns metric lists."""
    h5_files = sorted(Path(val_dir).glob('*.h5'))
    if max_volumes:
        h5_files = h5_files[:max_volumes]

    ifft_psnrs, ifft_ssims, unet_psnrs, unet_ssims = [], [], [], []

    model.eval()
    with torch.no_grad():
        for h5_path in tqdm(h5_files, desc=f"Eval {acceleration}x"):
            try:
                with h5py.File(h5_path, 'r') as f:
                    n = f['kspace'].shape[0]
                    for s in range(n // 4, 3 * n // 4):
                        ks = torch.from_numpy(f['kspace'][s].copy())
                        tgt = torch.from_numpy(f['reconstruction_esc'][s].copy()).float()

                        mask = create_mask(ks.shape, acceleration=acceleration, seed=s)
                        ifft = center_crop(torch.abs(from_kspace(ks * mask)), (320, 320))
                        ifft_n = ((ifft - ifft.min()) / (ifft.max() - ifft.min() + 1e-8)).numpy()
                        tgt_n = ((tgt - tgt.min()) / (tgt.max() - tgt.min() + 1e-8)).numpy()

                        m = compute_metrics(tgt_n, ifft_n)
                        ifft_psnrs.append(m['psnr']); ifft_ssims.append(m['ssim'])

                        unet_out = model(torch.from_numpy(ifft_n).unsqueeze(0).unsqueeze(0).to(device)
                                        ).cpu().squeeze().numpy()
                        m = compute_metrics(tgt_n, unet_out)
                        unet_psnrs.append(m['psnr']); unet_ssims.append(m['ssim'])
            except Exception:
                continue

    return {'ifft_psnr': ifft_psnrs, 'ifft_ssim': ifft_ssims,
            'unet_psnr': unet_psnrs, 'unet_ssim': unet_ssims}


results_4x = evaluate_method('/content/data/singlecoil_val/', model, device, acceleration=4)
results_8x = evaluate_method('/content/data/singlecoil_val/', model, device, acceleration=8)

print(f"\n{'Method':<10} {'Accel':>5} {'PSNR (dB)':>16} {'SSIM':>16}")
print("-" * 55)
for label, acc, res in [('IFFT', '4x', results_4x), ('U-Net', '4x', results_4x),
                         ('IFFT', '8x', results_8x), ('U-Net', '8x', results_8x)]:
    k = 'ifft' if label == 'IFFT' else 'unet'
    p, s = res[f'{k}_psnr'], res[f'{k}_ssim']
    print(f"{label:<10} {acc:>5} {np.mean(p):>8.1f} +/- {np.std(p):<5.1f} "
          f"{np.mean(s):>8.3f} +/- {np.std(s):<5.3f}")

print(f"\nSlices evaluated: {len(results_4x['ifft_psnr'])}")
print(f"U-Net gain: +{np.mean(results_4x['unet_psnr'])-np.mean(results_4x['ifft_psnr']):.1f} dB at 4x, "
      f"+{np.mean(results_8x['unet_psnr'])-np.mean(results_8x['ifft_psnr']):.1f} dB at 8x")

In [ ]:
n_vols = len(set(p for p, _ in train_dataset.examples))

results_summary = {
    'training': {
        'n_train_volumes': n_vols,
        'n_train_slices': len(train_dataset),
        'epochs': len(train_losses),
        'best_epoch': int(np.argmin(val_losses) + 1),
        'best_val_loss': float(min(val_losses)),
    },
    '4x': {k: float(v) for k, v in {
        'ifft_psnr_mean': np.mean(results_4x['ifft_psnr']),
        'ifft_psnr_std': np.std(results_4x['ifft_psnr']),
        'ifft_ssim_mean': np.mean(results_4x['ifft_ssim']),
        'unet_psnr_mean': np.mean(results_4x['unet_psnr']),
        'unet_psnr_std': np.std(results_4x['unet_psnr']),
        'unet_ssim_mean': np.mean(results_4x['unet_ssim']),
    }.items()},
    '8x': {k: float(v) for k, v in {
        'ifft_psnr_mean': np.mean(results_8x['ifft_psnr']),
        'ifft_psnr_std': np.std(results_8x['ifft_psnr']),
        'ifft_ssim_mean': np.mean(results_8x['ifft_ssim']),
        'unet_psnr_mean': np.mean(results_8x['unet_psnr']),
        'unet_psnr_std': np.std(results_8x['unet_psnr']),
        'unet_ssim_mean': np.mean(results_8x['unet_ssim']),
    }.items()},
}

with open('/content/drive/MyDrive/fastmri/checkpoints/nb02_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)
!cp figures/02_*.png /content/drive/MyDrive/fastmri/checkpoints/ 2>/dev/null || true

print("Results saved to Drive.\n")

## Summary

| Method | Accel | PSNR (dB) | SSIM |
|--------|-------|-----------|------|
| IFFT | 4x | 23.3 ± 3.1 | 0.569 ± 0.090 |
| U-Net | 4x | 26.2 ± 2.4 | 0.599 ± 0.118 |
| IFFT | 8x | 22.3 ± 3.2 | 0.482 ± 0.110 |
| U-Net | 8x | 24.4 ± 2.6 | 0.515 ± 0.125 |

The U-Net gains +2.9 dB at 4x and +2.1 dB at 8x over zero-filled IFFT. This is consistent with the expected performance for a model trained on 150 of 973 available volumes. The full training set would yield ~30 to 34 dB at 4x (Zbontar et al., 2018), but moderate performance is preferable here: a weaker model produces stronger hallucination signals for the detection methods in Notebooks 03 through 05.

Peck et al. (2026) demonstrated that even high-quality reconstructions (30+ dB) contain hallucinations invisible to PSNR and SSIM. The question is not whether our model hallucinates, but whether we can detect where.

**Next:** Notebook 03 runs the Bhadra et al. (2021) null-space decomposition on U-Net reconstructions to characterize the spatial distribution and frequency content of hallucinations.

**Checkpoint:** `unet_4x_v2_best.pt` on Google Drive.